# TA-5：RoBERTa 情感微调（Colab）

**运行前请确认（按需改下一格里的变量）：**

1. `REPO_URL`：已默认 `KevinL35/RSA`；换仓库时再改第二格。
2. `DRIVE_SPLITS`：Google Drive 里三个 CSV 所在目录（已默认 `.../RSA/finetune/data/splits`）。若你的数据在别的路径，只改第二格。

**数据文件：** `train.csv`、`val.csv`、`test.csv`，列需含 `analysis_input_en`、`label_sentiment`（0/1/2）。

**运行时：** 菜单「运行时 → 更改运行时类型 → GPU」建议选 T4 或更好。

In [ ]:
# ============ 按你的实际情况修改 ============
REPO_URL = "https://github.com/KevinL35/RSA.git"
DRIVE_SPLITS = "/content/drive/MyDrive/RSA/finetune/data/splits"  # 你的三个 CSV 在 Drive 上的目录
# ==========================================

import os
REPO_DIR = "/content/RSA"

## 1) 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2) 克隆代码（首次）或拉取更新

In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("OK:", REPO_DIR)

## 3) 安装依赖

In [ ]:
%cd /content/RSA
!pip install -q -r ml/requirements-finetune.txt

## 4) 把 Drive 里的划分文件拷到仓库 `ml/data/splits/`

若你**不用 Drive**、而是把三个 CSV 手动上传到 Colab 左侧并放到 `RSA/ml/data/splits/`，可**跳过本格**，直接跑下一格检查。

In [ ]:
import shutil

dst = os.path.join(REPO_DIR, "ml/data/splits")
os.makedirs(dst, exist_ok=True)

for name in ("train.csv", "val.csv", "test.csv"):
    src = os.path.join(DRIVE_SPLITS, name)
    if not os.path.isfile(src):
        raise FileNotFoundError(f"找不到: {src} 请检查 DRIVE_SPLITS 或文件名")
    shutil.copy2(src, os.path.join(dst, name))
    print("copied", name)

!ls -la {dst}

## 5) 确认文件存在

In [ ]:
!ls -la /content/RSA/ml/data/splits/

## 6) 训练（TA-5）

In [ ]:
%cd /content/RSA
!python ml/scripts/train_sentiment.py --config ml/configs/train_roberta_colab.yaml

## 7)（可选）仅在测试集上再评一次

In [ ]:
%cd /content/RSA
!python ml/scripts/evaluate_sentiment.py \
  --config ml/configs/train_roberta_colab.yaml \
  --checkpoint_dir ml/artifacts/roberta-sentiment-v0-colab

## 8)（建议）把模型产物拷回 Drive，避免断开连接后丢失

可改 `BACKUP` 目录名。

In [ ]:
import os
import shutil

BACKUP = "/content/drive/MyDrive/RSA/ml_artifacts_colab"
os.makedirs(BACKUP, exist_ok=True)
art = "/content/RSA/ml/artifacts/roberta-sentiment-v0-colab"
dst_art = os.path.join(BACKUP, "roberta-sentiment-v0-colab")
if os.path.isdir(dst_art):
    shutil.rmtree(dst_art)
shutil.copytree(art, dst_art)
rep = "/content/RSA/ml/reports/sentiment_eval_v0_colab.json"
if os.path.isfile(rep):
    shutil.copy2(rep, BACKUP)
print("备份到:", BACKUP)